# 02 — TensorRT Inference Sweep

For each trained checkpoint (`seed_1`, `seed_2`, `seed_42`), sweep all 5 precisions × 4 input bit-depths.

ONNX exports (base + QDQ) are assumed to already exist from `00_qdq_export.ipynb`.

Results saved under `runs/val_infer/<seed>/`, averaged at the end.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

In [2]:
import torch
import pandas as pd

from src.config import ExperimentConfig, with_overrides
from src.runner import run_experiment
from utils.utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [3]:
CKPT_ROOT = PROJECT_ROOT / "training" / "checkpoints" / "fp32"
SEEDS = sorted([p.name for p in CKPT_ROOT.iterdir() if p.is_dir() and (p / "best.pth").exists()])
checkpoints = {seed: str(CKPT_ROOT / seed / "best.pth") for seed in SEEDS}

INPUT_BITS = [8, 4, 2, 1]

def seed_num(seed):
    return seed.split("_")[-1]

def make_base(seed):
    prefix = f"resnet_{seed_num(seed)}"
    return ExperimentConfig(
        backend="tensorrt",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=500,
        trt_static_shape=True,
        trt_workspace_mb=2048,
        onnx_root=str(PROJECT_ROOT / "onnx"),
        engine_root=str(PROJECT_ROOT / "engines" / seed),
        trt_onnx_prefix=prefix,
        output_root=f"../runs/val_infer/{seed}",
    )

def run_precision_sweep(precision):
    records = []
    for seed, ckpt_path in checkpoints.items():
        print(f"\n  {seed}:")
        base = make_base(seed)
        for bits in INPUT_BITS:
            cfg = with_overrides(base, model_precision=precision, input_quant_bits=bits)
            payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=ckpt_path)
            records.append(payload)
            r = payload["results"]
            print(f"    b={bits}: top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")
    return records

print(f"Seeds: {SEEDS}")
print(f"Input bits: {INPUT_BITS}")

Seeds: ['seed_1', 'seed_2', 'seed_42']
Input bits: [8, 4, 2, 1]


## FP32

In [4]:
fp32_records = run_precision_sweep("fp32")


  seed_1:
[runner] Step 1/3 — ONNX exists, skipping: /home/pf4636/code/resnet/quantized_resnets/onnx/resnet_1.onnx
[runner] Step 2/3 — Engine exists, skipping: /home/pf4636/code/resnet/quantized_resnets/engines/seed_1/resnet18_tensorrt_fp32_in8b_cuda_bs1.engine
[runner] Step 3/3 — Running TRT inference ...
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/seed_1/resnet18_tensorrt_fp32_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 500 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 1.02 ms/batch
  [50/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 1.00 ms/batch
  [60/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 1.02 ms/batch
  [70/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 1.02 ms/batch
  [80/500]  To

## FP16

In [5]:
fp16_records = run_precision_sweep("fp16")


  seed_1:
[runner] Step 1/3 — ONNX exists, skipping: /home/pf4636/code/resnet/quantized_resnets/onnx/resnet_1.onnx
[runner] Step 2/3 — Building TRT engine (precision=fp16) ...
[trt_builder] Building engine | precision=fp16 | batch=1 | workspace=2048 MiB
[trt_builder] Engine saved: /home/pf4636/code/resnet/quantized_resnets/engines/seed_1/resnet18_tensorrt_fp16_in8b_cuda_bs1.engine (22.8 MB)
[runner] Step 3/3 — Running TRT inference ...
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/seed_1/resnet18_tensorrt_fp16_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 500 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 0.58 ms/batch
  [50/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 0.96 ms/batch
  [60/500]  

## INT8

In [6]:
int8_records = run_precision_sweep("int8")


  seed_1:
[runner] Step 1/3 — ONNX exists, skipping: /home/pf4636/code/resnet/quantized_resnets/onnx/resnet_1_int8_qdq.onnx
[runner] Step 2/3 — Building TRT engine (precision=int8) ...
[trt_builder] Building engine | precision=int8 | batch=1 | workspace=2048 MiB
[trt_builder] Engine saved: /home/pf4636/code/resnet/quantized_resnets/engines/seed_1/resnet18_tensorrt_int8_in8b_cuda_bs1.engine (11.7 MB)
[runner] Step 3/3 — Running TRT inference ...
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/seed_1/resnet18_tensorrt_int8_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 500 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 0.72 ms/batch
  [50/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 0.90 ms/batch
  [

## FP8

In [7]:
fp8_records = run_precision_sweep("fp8")


  seed_1:
[runner] Step 1/3 — ONNX exists, skipping: /home/pf4636/code/resnet/quantized_resnets/onnx/resnet_1_fp8_qdq.onnx
[runner] Step 2/3 — Building TRT engine (precision=fp8) ...
[trt_builder] Building engine | precision=fp8 | batch=1 | workspace=2048 MiB
[trt_builder] Engine saved: /home/pf4636/code/resnet/quantized_resnets/engines/seed_1/resnet18_tensorrt_fp8_in8b_cuda_bs1.engine (11.7 MB)
[runner] Step 3/3 — Running TRT inference ...
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/seed_1/resnet18_tensorrt_fp8_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 500 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 1.01 ms/batch
  [50/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 0.89 ms/batch
  [60/50

## INT4

In [8]:
int4_records = run_precision_sweep("int4")


  seed_1:
[runner] Step 1/3 — ONNX exists, skipping: /home/pf4636/code/resnet/quantized_resnets/onnx/resnet_1_int4_qdq.onnx
[runner] Step 2/3 — Building TRT engine (precision=int4) ...
[trt_builder] Building engine | precision=int4 | batch=1 | workspace=2048 MiB
[trt_builder] Engine saved: /home/pf4636/code/resnet/quantized_resnets/engines/seed_1/resnet18_tensorrt_int4_in8b_cuda_bs1.engine (22.9 MB)
[runner] Step 3/3 — Running TRT inference ...
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/seed_1/resnet18_tensorrt_int4_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 500 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 0.52 ms/batch
  [50/500]  Top-1: 100.00%  Top-5: 100.00%  Infer: 0.51 ms/batch
  [

## All Results (per seed)

In [9]:
all_rows = []
for seed in SEEDS:
    runs = load_runs(f"../runs/val_infer/{seed}", status="ok")
    for r in flatten_runs(runs):
        r["seed"] = seed
        all_rows.append(r)

df = pd.DataFrame(all_rows)
df_trt = df[df["cfg.backend"] == "tensorrt"]

display_cols = [
    "seed", "cfg.model_precision", "cfg.input_quant_bits",
    "res.top1_acc", "res.top5_acc", "res.infer_ms_avg", "res.throughput_infer_sps",
]
df_trt[display_cols].sort_values(
    ["seed", "cfg.model_precision", "cfg.input_quant_bits"], ascending=[True, True, False]
).reset_index(drop=True)

,seed,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps
0,seed_1,fp16,8,98.936,100.000,0.488,2048.288
1,seed_1,fp16,4,97.234,100.000,0.504,1984.305
2,seed_1,fp16,2,58.936,82.128,0.491,2035.892
3,seed_1,fp16,1,33.617,60.638,0.495,2021.576
4,seed_1,fp32,8,98.936,100.000,0.953,1049.026
5,seed_1,fp32,4,97.234,100.000,0.951,1051.343
6,seed_1,fp32,2,58.936,82.128,0.955,1047.381
7,seed_1,fp32,1,33.617,60.638,0.946,1056.946
8,seed_1,fp8,8,98.936,100.000,0.667,1500.255
9,seed_1,fp8,4,97.234,100.000,0.672,1488.840


## Averaged Results (across seeds)

In [14]:
avg_df = df_trt.groupby(["cfg.backend", "cfg.model_precision", "cfg.input_quant_bits"]).agg(
    top1_mean=("res.top1_acc", "mean"),
    top1_std=("res.top1_acc", "std"),
    top5_mean=("res.top5_acc", "mean"),
    top5_std=("res.top5_acc", "std"),
    infer_ms_mean=("res.infer_ms_avg", "mean"),
    infer_ms_std=("res.infer_ms_avg", "std"),
    throughput_mean=("res.throughput_infer_sps", "mean"),
).round(3).reset_index()

avg_df = avg_df.sort_values(["cfg.model_precision", "cfg.input_quant_bits"], ascending=[True, False]).reset_index(drop=True)
avg_df

,cfg.backend,cfg.model_precision,cfg.input_quant_bits,top1_mean,top1_std,top5_mean,top5_std,infer_ms_mean,infer_ms_std,throughput_mean
0,tensorrt,fp16,8,94.397,7.137,99.574,0.737,0.478,0.018,2092.942
1,tensorrt,fp16,4,93.191,6.457,99.291,1.228,0.490,0.012,2041.567
2,tensorrt,fp16,2,56.028,3.820,80.355,1.931,0.497,0.016,2013.420
3,tensorrt,fp16,1,32.199,1.494,58.865,1.931,0.504,0.013,1985.312
4,tensorrt,fp32,8,94.468,7.015,99.574,0.737,0.994,0.082,1010.735
5,tensorrt,fp32,4,93.191,6.457,99.291,1.228,0.997,0.065,1005.734
6,tensorrt,fp32,2,55.957,3.940,80.355,1.931,0.961,0.060,1043.073
7,tensorrt,fp32,1,32.270,1.384,58.794,1.919,0.964,0.038,1038.594
8,tensorrt,fp8,8,94.255,7.561,99.574,0.737,0.662,0.009,1510.908
9,tensorrt,fp8,4,92.766,7.015,99.149,1.474,0.668,0.006,1496.551


In [15]:
avg_df[["cfg.model_precision", "cfg.input_quant_bits",
        "top1_mean", "top1_std", "top5_mean", "top5_std",
        "infer_ms_mean", "infer_ms_std"]].assign(
    top1=lambda d: d.apply(lambda r: f"{r['top1_mean']:.2f} ± {r['top1_std']:.2f}", axis=1),
    top5=lambda d: d.apply(lambda r: f"{r['top5_mean']:.2f} ± {r['top5_std']:.2f}", axis=1),
    infer_ms=lambda d: d.apply(lambda r: f"{r['infer_ms_mean']:.3f} ± {r['infer_ms_std']:.3f}", axis=1),
)[["cfg.model_precision", "cfg.input_quant_bits", "top1", "top5", "infer_ms"]]

,cfg.model_precision,cfg.input_quant_bits,top1,top5,infer_ms
0,fp16,8,94.40 ± 7.14,99.57 ± 0.74,0.478 ± 0.018
1,fp16,4,93.19 ± 6.46,99.29 ± 1.23,0.490 ± 0.012
2,fp16,2,56.03 ± 3.82,80.36 ± 1.93,0.497 ± 0.016
3,fp16,1,32.20 ± 1.49,58.87 ± 1.93,0.504 ± 0.013
4,fp32,8,94.47 ± 7.01,99.57 ± 0.74,0.994 ± 0.082
5,fp32,4,93.19 ± 6.46,99.29 ± 1.23,0.997 ± 0.065
6,fp32,2,55.96 ± 3.94,80.36 ± 1.93,0.961 ± 0.060
7,fp32,1,32.27 ± 1.38,58.79 ± 1.92,0.964 ± 0.038
8,fp8,8,94.25 ± 7.56,99.57 ± 0.74,0.662 ± 0.009
9,fp8,4,92.77 ± 7.01,99.15 ± 1.47,0.668 ± 0.006


In [16]:
out_path = PROJECT_ROOT / "results" / "tensorrt_avg_results.json"
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")

Saved to /home/pf4636/code/resnet/quantized_resnets/results/tensorrt_avg_results.json
